In [ ]:
import cv2
import glob
import numpy as np
import pandas as pd
import scipy.io
from scipy.ndimage import rotate
from skimage import transform as tf
from skimage.transform import warp
import scipy.io
import cv2
import matplotlib.pyplot as plt
import scanpy as sc

sample = 'mouse_skin'

all_cell_mapping = pd.read_csv(f'./{sample}/all_cell_mapping.csv', index_col=0)
all_cell_raman = pd.read_csv(f'./{sample}/all_cell_raman.csv', index_col=0)
all_cell_imputation = pd.read_csv(f'./{sample}/all_cell_imputation_filtered.csv', index_col=0)

In [ ]:
import scanpy as sc
adata_imputed = sc.AnnData(all_cell_imputation.values)
adata_imputed.obs_names = all_cell_imputation.index
adata_imputed.var_names = all_cell_imputation.columns
# normalize
sc.pp.normalize_total(adata_imputed, target_sum=1e4)
sc.pp.log1p(adata_imputed)
all_cell_imputation = pd.DataFrame(adata_imputed.X, index=adata_imputed.obs_names, columns=adata_imputed.var_names)
all_cell_imputation

In [ ]:
from scipy.integrate import simpson
from scipy.stats import ranksums
from statsmodels.stats.multitest import multipletests
import seaborn as sns
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score
from matplotlib.colors import LinearSegmentedColormap


wave_number = scipy.io.loadmat('./wavenumbers.mat')['wavenumber'][0][413:1286].round(0).astype(np.int16)

def normalize_spectra(wavenum, intensity, min_peak, max_peak):
    amide_mask = (wavenum >= min_peak) & (wavenum <= max_peak)
    area = simpson(intensity[amide_mask], x=wavenum[amide_mask])
    # print(area)
    return intensity / area

def calculate_DEP(data1, data2, wave_number, label_set, cell_type, prefix):

    import numpy as np
    import pandas as pd
    import scanpy as sc

    # Calculate the Wilcoxon rank-sum test for each feature (column)
    p_values = []
    log2FC = []
    for column in data1.columns:
        stat, p_value = ranksums(data1[column], data2[column])
        p_values.append(p_value)

        mean_data1 = data1[column].mean()
        mean_data2 = data2[column].mean()
        log2FC.append(np.log2(mean_data1 / mean_data2))
        
    # Apply FDR correction
    _, p_values_corrected, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')

    # Create a DataFrame with the original and corrected p-values
    results = pd.DataFrame({
            'Index': range(len(data1.columns)),
            'Peak': data1.columns,
            'p-value': p_values,
            'p-value_corrected': p_values_corrected,
            'log2FC': log2FC
        })

    # Filter features with FDR < 0.05
    significant_results = results[results['p-value_corrected'] <= 0.05]

    upregulated = significant_results[significant_results['log2FC'] > 0]
    downregulated = significant_results[significant_results['log2FC'] < 0]

    upregulated = upregulated.sort_values('log2FC', ascending=False)
    downregulated = downregulated.sort_values('log2FC', ascending=True)
    
    if len(upregulated) < 1 or len(downregulated) < 1:
        return None, None
     
    # Print significant features
    # print('Number of DEFs:', len(significant_results), 'upregulated:', len(upregulated), 'downregulated:', len(downregulated))
    else:
        print(label_set, cell_type, 'Up:', len(upregulated), 'Down:', len(downregulated))
        # highlight down- or up- regulated genes
        down = results[(results['log2FC']<=0)&(results['p-value_corrected']<=0.05)]
        up = results[(results['log2FC']>=0)&(results['p-value_corrected']<=0.05)]

        # mark the top 3 up and down regulated genes consider log2FC 
        up = up.sort_values('log2FC',ascending=False)
        down = down.sort_values('log2FC',ascending=True)

        return up['Peak'].values[:30], down['Peak'].values[:30]


def random_forest(combined, raman, gene, labels, label_set, cell_type, prefix):
    
    seed = 2024

    # downsample the majority class
    from imblearn.under_sampling import RandomUnderSampler
    rus = RandomUnderSampler(random_state=seed)
    combined, labels = rus.fit_resample(combined, labels)
    print('Subsampled integrated features::', combined.shape)

    # train test split
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(combined, labels, test_size=0.3, random_state=seed)

    # z-score
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_train_norm = scaler.fit_transform(X_train.values)
    X_test_norm = scaler.transform(X_test.values)
    X_train = pd.DataFrame(X_train_norm, columns=X_train.columns)
    X_test_raw = pd.DataFrame(X_test, columns=X_test.columns)
    X_test = pd.DataFrame(X_test_norm, columns=X_test.columns)

    # train
    from sklearn.ensemble import RandomForestClassifier
    clf = RandomForestClassifier(n_estimators=500, random_state=seed)  
    clf_raman = RandomForestClassifier(n_estimators=500, random_state=seed)     
    clf_gene = RandomForestClassifier(n_estimators=500, random_state=seed)      
    clf.fit(X_train, y_train)
    clf_raman.fit(X_train.iloc[:, :raman.shape[1]], y_train)
    clf_gene.fit(X_train.iloc[:, raman.shape[1]:], y_train)

    y_pred = clf.predict(X_test)
    y_pred_raman = clf_raman.predict(X_test.iloc[:, :raman.shape[1]])
    y_pred_gene = clf_gene.predict(X_test.iloc[:, raman.shape[1]:])

    y_prob_all = clf.predict_proba(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1] 
    y_prob_raman = clf_raman.predict_proba(X_test.iloc[:, :raman.shape[1]])[:, 1]
    y_prob_gene = clf_gene.predict_proba(X_test.iloc[:, raman.shape[1]:])[:, 1] 

    print('Accuracy: {:.4f} {:.4f} {:.4f}'.format(accuracy_score(y_test, y_pred), accuracy_score(y_test, y_pred_gene), accuracy_score(y_test, y_pred_raman)))
    print('ROC AUC: {:.4f} {:.4f} {:.4f}'.format(roc_auc_score(y_test, y_prob), roc_auc_score(y_test, y_prob_gene), roc_auc_score(y_test, y_prob_raman)))
    print('F1: {:.4f} {:.4f} {:.4f}'.format(f1_score(y_test, y_pred), f1_score(y_test, y_pred_gene), f1_score(y_test, y_pred_raman)))
    print('Precision: {:.4f} {:.4f} {:.4f}'.format(precision_score(y_test, y_pred), precision_score(y_test, y_pred_gene), precision_score(y_test, y_pred_raman)))
    print('Recall: {:.4f} {:.4f} {:.4f}'.format(recall_score(y_test, y_pred), recall_score(y_test, y_pred_gene), recall_score(y_test, y_pred_raman)))

    # Metrics to plot
    metrics = ['Accuracy', 'AUC', 'F1 Score', 'Precision', 'Recall']
    values = [accuracy_score(y_test, y_pred), roc_auc_score(y_test, y_prob), f1_score(y_test, y_pred), precision_score(y_test, y_pred), recall_score(y_test, y_pred)]
    values_gene = [accuracy_score(y_test, y_pred_gene), roc_auc_score(y_test, y_prob_gene), f1_score(y_test, y_pred_gene), precision_score(y_test, y_pred_gene), recall_score(y_test, y_pred_gene)]
    values_raman = [accuracy_score(y_test, y_pred_raman), roc_auc_score(y_test, y_prob_raman), f1_score(y_test, y_pred_raman), precision_score(y_test, y_pred_raman), recall_score(y_test, y_pred_raman)]
        
    # Number of metrics
    n_metrics = len(metrics)
    # X-axis locations for the groups
    x = np.arange(n_metrics)
    # Width of the bars
    bar_width = overlap_shift = 0.3
    # Plotting the overlapping bar chart with 50% overlap
    plt.figure(figsize=(6, 4))
    
    plt.bar(x - overlap_shift, values_raman, width=bar_width, label='Raman', color='#B3CDE4', alpha=0.8)  # blue D3E6F4
    plt.bar(x, values_gene, width=bar_width, label='snRNA-seq', color='#EB928C', alpha=0.8)   # red F0AAA1
    plt.bar(x + overlap_shift, values, width=bar_width, label='Multimodal', color='#808285', alpha=0.8)  # green  CDE7A8

    # Add labels and titles
    # plt.xlabel('Metrics')
    plt.ylabel('Score')
    plt.title('Comparison of performance across different modalities for ' + label_set + ' (' + cell_type + ')')
    plt.xticks(x, metrics)

    # Add legend
    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1))
    plt.rcParams.update({'font.size': 8})
    # set frnt style
    plt.rcParams.update({'font.family': 'Arial'})

    # set y lim
    if label_set == 'p21+':
        plt.ylim(0.2, 0.9)
    elif label_set == 'SenMayo':
        plt.ylim(0.2, 0.7)

    # hide the right and top spines
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)

    # Display the plot
    plt.tight_layout()
    # save
    plt.savefig('./' + prefix + '_performance_' + label_set + '_' + cell_type + '.pdf', bbox_inches='tight') # dpi 500
    plt.show()

    # feature importance
    importances = clf.feature_importances_
    indices = np.argsort(importances)[::-1]
    names = [combined.columns[i] for i in indices]
    # plot the top 30 features and their importance
    plt.figure(figsize=(12, 6))
    plt.rcParams.update({'font.size': 8})
    # set frnt style
    plt.rcParams.update({'font.family': 'Arial'})
    plt.title("Top 30 feature importances for " + label_set + " (" + cell_type + ")")

    # set gene features to be blue raman to be red
    colors = ['#D3E6F4'] * raman.shape[1] + ['#F0AAA1'] * gene.shape[1]
    # set the color of the top 30 features
    for i in range(30):
        plt.bar(i, importances[indices[i]], color=colors[indices[i]], align="center")
    # set the legend, blue for gene features, red for raman features
    legend_color_mapping = {'scRNA-seq': '#F0AAA1', 'Raman': '#D3E6F4'}
    legend_labels = ['scRNA-seq', 'Raman']
    legend_handles = [plt.Rectangle((0,0),1,1, color=legend_color_mapping[label]) for label in legend_labels]
    plt.legend(legend_handles, legend_labels)
    # set the xticks and xlabels

    plt.xticks(range(30), names[:30], rotation=45)
    plt.xlim([-1, 30])
    # replace X as the feature names
    # plt.xlabel('Top 30 multi-modal features')
    plt.ylabel('Importance score')
    # hide the right and top spines
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)
    # save
    plt.savefig('./' + prefix + '_feature_importance_' + label_set + '_' + cell_type + '.pdf', bbox_inches='tight')
    plt.show()

    import shap
    explainer = shap.TreeExplainer(clf, X_test)
    shap_values = explainer(X_test)

    def plot_barcode_comparison(ranked_features, features, gene, group1_values, group2_values, label_set, cell_type, prefix):
        features_type = [True if f in gene else False for f in ranked_features]

        sorted_indices = np.argsort(group1_values)
        sorted_features = [ranked_features[i] for i in sorted_indices]
        print(sorted_features)

        sorted_indices = np.argsort(group2_values) # from small to large
        sorted_features = [ranked_features[i] for i in sorted_indices]
        print(sorted_features)

        print('number of features for barcode:', len(ranked_features), len(group1_values), len(group2_values))
        # Create a figure and axis
        fig, ax = plt.subplots(figsize=(10, 3))

        # Plot the first array as a barcode
        for i in range(len(group1_values)):
            color = '#EB382E' if features_type[i] else '#91B9D6'
            ax.plot([group1_values[i], group1_values[i]], [0, 1], color=color, linewidth=1, alpha=0.7)
            ax.text(group1_values[i], 1.2, ranked_features[i], ha='center', va='center', fontsize=1, color='black', rotation=90)

        # Plot the second array as a barcode (offset the lines slightly to avoid overlap)
        for i in range(len(group2_values)):
            color = '#EB382E' if features_type[i] else '#91B9D6' 
            ax.plot([group2_values[i], group2_values[i]], [1.6, 2.6], color=color, linewidth=1, alpha=0.7)
            ax.text(group2_values[i], 3, ranked_features[i], ha='center', va='center', fontsize=1, color='black', rotation=90)

        # Add labels and title
        ax.set_yticks([0.5, 1.6])
        ax.set_yticklabels(['Senescence', 'Non-senescence'])

        # Remove the spines
        plt.gca().spines['left'].set_visible(False)
        plt.gca().spines['right'].set_visible(False)
        plt.gca().spines['top'].set_visible(False)
        plt.gca().spines['bottom'].set_visible(False)

        plt.grid(False)
        plt.title(f'{label_set} ({cell_type})')  # Add a title

        # tight layout
        plt.tight_layout()

        ax.set_xlim(0, 0.1)
        plt.savefig(f'./{prefix}_{label_set}_{cell_type}_0_0.1.pdf')
        ax.set_xlim(0.1, 5)
        
        plt.savefig(f'./{prefix}_{label_set}_{cell_type}_0.1_5.pdf')

        plt.show()

        print('barcodes len', len(ranked_features), len(group1_values), len(group2_values))

        return group1_values, group2_values, ranked_features


    ranked_index = np.argsort(np.abs(shap_values.values[:, :, 1]).mean(axis=0))[::-1]
    ranked_columns = [X_test.columns[i] for i in ranked_index]
    
    number_of_selected_features = 60
    selected_X_test = X_test_raw[ranked_columns[:number_of_selected_features]] # top 60 and ranked features

    mean_data1 = selected_X_test[y_test == 1].mean(0)
    mean_data2 = selected_X_test[y_test == 0].mean(0)

    ranked_columns = ranked_columns[:number_of_selected_features]
    print('Selected features:', ranked_columns)

    group1_values, group2_values, ranked_features = plot_barcode_comparison(ranked_columns, selected_X_test, gene.columns, mean_data1.values, mean_data2.values, label_set, cell_type, prefix)


    num_colors = 10
    newCmap = LinearSegmentedColormap.from_list("custom_cmap", ['#A3C4D3','#D9726C'])
    color_list = [newCmap(i/num_colors) for i in range(num_colors)]
    plt.rcParams['font.size'] = 8  # Set font size (you can adjust the number)
    plt.rcParams['font.family'] = 'Arial'  # Set font family (you can adjust the font)
    shap.summary_plot(shap_values[:, :, 1], color=color_list,  max_display=30, cmap=newCmap,show=False)
    plt.tight_layout()
    plt.savefig('./' + prefix + '_shap_summary_bar_' + label_set + '_' + cell_type + '.pdf', bbox_inches='tight')
    plt.close()

    num_colors = 2
    newCmap = LinearSegmentedColormap.from_list("custom_cmap", ['#A3C4D3', '#A3C4D3'])
    color_list = [newCmap(i/num_colors) for i in range(num_colors)]
    shap.summary_plot(shap_values[:, :, 1], color=color_list, max_display=30,  show=False, plot_type='bar', cmap=newCmap)
    plt.tight_layout()
    plt.savefig('./' + prefix + '_shap_summary_' + label_set + '_' + cell_type + '.pdf', bbox_inches='tight')
    plt.close()

    from shap.plots import _style
    with _style.style_context(
        primary_color_positive="#D9726C",
        primary_color_negative="#A3C4D3",
    ):

        # High confidence for class 1 (senescent)
        high_conf_senescent_idx = np.argmax(y_prob_all[:, 1])  # Index of the highest probability for class 1
        # High confidence for class 0 (non-senescent)
        high_conf_non_senescent_idx = np.argmax(y_prob_all[:, 0])  # Index of the highest probability for class 0
        # Find borderline samples (predicted probability close to 0.5 for both classes)
        borderline_idx = np.argmin(np.abs(y_prob_all[:, 1] - 0.5))  # Sample with the closest probability to 0.5
        shap_values_senescent = shap_values[:, :, 1][high_conf_senescent_idx]
        shap_values_non_senescent = shap_values[:, :, 1][high_conf_non_senescent_idx]
        shap_values_borderline = shap_values[:, :, 1][borderline_idx]
        plt.rcParams['font.size'] = 8  # Set font size (you can adjust the number)
        plt.rcParams['font.family'] = 'Arial'  # Set font family (you can adjust the font)

        fig = plt.figure()
        shap.plots.waterfall(shap_values_senescent, max_display=20, show=False)
        plt.tight_layout()
        plt.rcParams.update({'font.size': 8})
        # set frnt style
        plt.rcParams.update({'font.family': 'Arial'})
        plt.savefig('./' + prefix + '_shap_waterfall_senescent_' + label_set + '_' + cell_type + '.pdf', bbox_inches='tight')
        plt.close()

        fig = plt.figure()
        shap.plots.waterfall(shap_values_non_senescent, max_display=20, show=False)
        plt.tight_layout()
        plt.rcParams.update({'font.size': 8})
        # set frnt style
        plt.rcParams.update({'font.family': 'Arial'})
        plt.savefig('./' + prefix + '_shap_waterfall_non_senescent_' + label_set + '_' + cell_type + '.pdf', bbox_inches='tight')
        plt.close()

        fig = plt.figure()
        shap.plots.waterfall(shap_values_borderline, max_display=20, show=False)
        plt.tight_layout()
        plt.rcParams.update({'font.size': 8})
        # set frnt style
        plt.rcParams.update({'font.family': 'Arial'})
        plt.savefig('./' + prefix + '_shap_waterfall_borderline_' + label_set + '_' + cell_type + '.pdf', bbox_inches='tight')
        plt.close()

    correlation_matrix = X_test.corr()
    # Define gene and Raman feature names
    gene_feature_names = gene.columns
    raman_feature_names = raman.columns
    correlation_between_gene_and_raman = correlation_matrix.loc[gene_feature_names, raman_feature_names]

    fig_size = (15, 15)
    print('number of gene features:', len(gene_feature_names), 'number of raman features:', len(raman_feature_names))

    # Increase the figure size to fit all feature names
    g = sns.clustermap(correlation_between_gene_and_raman, cmap='coolwarm', annot=False, square=False,
                    method='average', metric='correlation', linewidths=1, figsize=fig_size)

    # Rotate the labels for better visibility
    plt.setp(g.ax_heatmap.get_xticklabels(), rotation=90)  # Rotate x-axis labels (Raman features)
    plt.setp(g.ax_heatmap.get_yticklabels(), rotation=0)   # Keep y-axis labels (Gene features) horizontal
    plt.rcParams.update({'font.size': 8})
        # set frnt style
    plt.rcParams.update({'font.family': 'Arial'})
    plt.title('Correlation Between scRNA-seq and Raman features for ' + label_set + ' (' + cell_type + ')')
    # save
    plt.savefig('./' + prefix + '_correlation_' + label_set + '_' + cell_type + '.pdf', bbox_inches='tight')
    plt.show()

    return group1_values, group2_values, ranked_features

In [ ]:
p21_old = [
    "rgs6", "tmeff2", "prickle2", "igfbp5", "lmntd1", "htr2c", "edil3", "piezo2", "chil3",
    "igfbp7", "kitl", "gpm6a", "atp6v0d2", "gm30382", "tbx3os1", "kcnip4", "mertk", "car4", "kalrn",
    "abcg1", "aff3", "mrc1", "prx", "dab2", "tbx3", "ccl6", "creb5", "dach1", "c530008m17rik",
    "dock10", "sulf1", "kcnk2", "por", "tmem132d", "osbpl6", "ccdc129", "errfi1", "alcam", "fmo3",
    "ank3", "limch1", "acoxl", "nebl", "col23a1", "mecom"
]

from scipy import stats
import copy

label_sets = [ 'p21+']
tiltle_sets = ['p21']

# global
for label_set,tiletle_set in zip(label_sets, tiltle_sets):

    selected_cell_mapping = all_cell_mapping[all_cell_mapping['sample_type'] == 'O']

    if len(selected_cell_mapping) < 3:
        continue

    selected_cell_raman = all_cell_raman.loc[selected_cell_mapping.index]

    labels = selected_cell_mapping[label_set].values * 1
    features = copy.deepcopy(selected_cell_raman)
    
    new_features = []
    for i in range(features.shape[0]):
        new_features.append(normalize_spectra(wave_number, features.values[i], 1630, 1700))
    new_features = np.stack(new_features, axis=0)
    features.values[:,:] = new_features
    
    positive = features[labels == 1]
    negative = features[labels == 0]
    
    if len(positive) < 3 or len(negative) < 3:
        continue

    up_peaks, down_peaks = calculate_DEP(positive, negative, wave_number, label_set, 'global', 'SvsNS')

    if up_peaks is None or down_peaks is None:
        continue

    up_raman_features = features[up_peaks]
    down_raman_features = features[down_peaks]
    combined_raman_features = pd.concat([up_raman_features, down_raman_features], axis=1)

    combined_gene_features = all_cell_imputation.loc[selected_cell_raman.index]

    combined_gene_features = combined_gene_features[p21_old]
    
    print('All Raman features:', combined_raman_features.shape)
    print('All scRNA-seq features:', combined_gene_features.shape)

    combined_integrated_features = pd.concat([combined_raman_features, combined_gene_features], axis=1)
    print('Integrated features:', combined_integrated_features.shape)

    random_forest(combined_integrated_features, combined_raman_features, combined_gene_features, labels, label_set, 'global', 'SvsNS')